# Creator Intelligence Platform EDA

This notebook analyzes the cleaned dataset stored in MySQL (`videos_cleaned`).

## Section 1 — Data Overview

In [1]:
import sys
import os
sys.path.insert(0, 'D:/creator-intelligence-platform')

from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from database.db_connection import get_mysql_connection

sns.set_style("whitegrid")
charts_dir = os.path.join("eda", "charts")
os.makedirs(charts_dir, exist_ok=True)

conn = get_mysql_connection()
try:
    df = pd.read_sql("SELECT * FROM videos_cleaned", conn)
finally:
    conn.close()


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22832\2514266747.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM videos_cleaned", conn)


## Section 2 — Engagement Analysis

In [2]:
if not df.empty:
    # Ensure numeric
    df['engagement_rate'] = pd.to_numeric(df['engagement_rate'], errors='coerce')

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.histplot(df['engagement_rate'], kde=True, ax=ax, color="#4E79A7")
    ax.set_title("Engagement Rate Distribution (Histogram + KDE)")
    ax.set_xlabel("Engagement Rate (%)")
    ax.set_ylabel("Frequency")

    median_val = float(df['engagement_rate'].median())
    ax.axvline(median_val, color="#F28E2B", linewidth=2)
    ax.annotate(f"Median: {median_val:.4f}%", xy=(median_val, ax.get_ylim()[1] * 0.9),
                xytext=(10, 0), textcoords='offset points', color="#F28E2B")

    out_path = os.path.join(charts_dir, "engagement_hist_kde.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)

    # Boxplot by category
    order = sorted(df['category'].dropna().unique().tolist())
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(data=df, x='category', y='engagement_rate', order=order, ax=ax, color="#59A14F")
    ax.set_title("Engagement Rate by Category (Boxplot)")
    ax.set_xlabel("Category")
    ax.set_ylabel("Engagement Rate (%)")

    # Annotate median values on the plot
    medians = df.groupby('category')['engagement_rate'].median().reindex(order)
    xticks = ax.get_xticks()
    for i, (cat, med) in enumerate(medians.items()):
        x = xticks[i] if i < len(xticks) else i
        ax.text(x, med, f"{med:.3f}", ha='center', va='bottom', fontsize=9, color='black')

    out_path = os.path.join(charts_dir, "engagement_boxplot_by_category.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)


Saved: eda\charts\engagement_hist_kde.png
Saved: eda\charts\engagement_boxplot_by_category.png


## Section 3 — Views Analysis

In [3]:
if not df.empty:
    df['view_count'] = pd.to_numeric(df['view_count'], errors='coerce')
    df['performance_tier'] = df['performance_tier'].astype(str)

    # Log-scale histogram of view_count
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.histplot(df['view_count'], bins=30, log_scale=True, ax=ax, color="#9C755F")
    ax.set_title("View Count Distribution (Log Scale)")
    ax.set_xlabel("View Count (log scale)")
    ax.set_ylabel("Frequency")

    median_views = float(df['view_count'].median())
    ax.axvline(median_views, color="#4E79A7", linewidth=2)
    ax.annotate(f"Median: {median_views:,.0f}", xy=(median_views, ax.get_ylim()[1] * 0.9),
                xytext=(10, 0), textcoords='offset points', color="#4E79A7")

    out_path = os.path.join(charts_dir, "view_count_log_hist.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)

    # Violin plot of views by performance_tier
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.violinplot(data=df, x='performance_tier', y='view_count', ax=ax, inner='quartile', color="#B07AA1")
    ax.set_title("Views by Performance Tier (Violin Plot)")
    ax.set_xlabel("Performance Tier")
    ax.set_ylabel("View Count")

    median_views_overall = float(df['view_count'].median())
    ax.annotate(f"Overall median: {median_views_overall:,.0f}", xy=(0.01, 0.98), xycoords='axes fraction',
                ha='left', va='top', fontsize=10, color='black')
    out_path = os.path.join(charts_dir, "views_violin_by_performance_tier.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)

    # Top 10 most viewed videos
    top10 = df.sort_values('view_count', ascending=False).head(10)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(top10['title'][::-1], top10['view_count'][::-1], color="#E15759")
    ax.set_title("Top 10 Most Viewed Videos")
    ax.set_xlabel("View Count")
    ax.set_ylabel("Video Title")

    for y_idx, val in enumerate(top10['view_count'][::-1].values):
        ax.text(val, y_idx, f"{val:,}", va='center', fontsize=9)

    fig.tight_layout()
    out_path = os.path.join(charts_dir, "top_10_most_viewed_videos.png")
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)


Saved: eda\charts\view_count_log_hist.png
Saved: eda\charts\views_violin_by_performance_tier.png
Saved: eda\charts\top_10_most_viewed_videos.png


## Section 4 — Upload Timing Analysis

In [4]:
if not df.empty:
    day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    df['upload_day'] = df['upload_day'].astype(str)

    # Heatmap: upload_hour vs upload_day vs avg engagement_rate
    agg = df.groupby(['upload_day', 'upload_hour'], as_index=False).agg(avg_engagement_rate=('engagement_rate', 'mean'))
    pivot = agg.pivot(index='upload_day', columns='upload_hour', values='avg_engagement_rate').reindex(day_order)

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(pivot, ax=ax, cmap='YlGnBu', annot=False)
    ax.set_title("Avg Engagement Rate: Upload Day vs Hour")
    ax.set_xlabel("Upload Hour")
    ax.set_ylabel("Upload Day")

    # Annotate the best (day, hour) cell on the heatmap
    max_val = float(np.nanmax(pivot.values))
    best_i, best_j = np.unravel_index(np.nanargmax(pivot.values), pivot.values.shape)
    best_day = pivot.index[best_i]
    best_hour = int(pivot.columns[best_j])
    ax.text(best_j + 0.5, best_i + 0.5, f"{max_val:.3f}", ha='center', va='center',
            color='black', fontsize=9, fontweight='bold')
    ax.annotate(f"Best: {best_day} @ {best_hour}:00", xy=(0.01, 0.98), xycoords='axes fraction',
                ha='left', va='top', fontsize=10, color='black')
    out_path = os.path.join(charts_dir, "upload_hour_vs_day_heatmap.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)

    # Line chart: avg views by upload_hour
    hourly_views = df.groupby('upload_hour', as_index=False).agg(avg_views=('view_count', 'mean')).sort_values('upload_hour')
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.lineplot(data=hourly_views, x='upload_hour', y='avg_views', marker='o', ax=ax, color="#4E79A7")
    ax.set_title("Average Views by Upload Hour")
    ax.set_xlabel("Upload Hour")
    ax.set_ylabel("Avg Views")

    # Annotate the best hour (max avg views)
    idx = int(hourly_views['avg_views'].idxmax())
    best_hour = int(hourly_views.loc[idx, 'upload_hour'])
    best_views = float(hourly_views.loc[idx, 'avg_views'])
    ax.annotate(f"Max: {best_hour}:00\n{best_views:,.0f} views", xy=(best_hour, best_views),
                xytext=(10, 10), textcoords='offset points', ha='left', fontsize=10, color='black')
    out_path = os.path.join(charts_dir, "avg_views_by_upload_hour.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)

    # Bar chart: video count by upload_day
    day_counts = df['upload_day'].value_counts().reindex(day_order).dropna().reset_index()
    day_counts.columns = ['upload_day', 'video_count']

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=day_counts, x='upload_day', y='video_count', ax=ax, color="#59A14F")
    ax.set_title("Video Count by Upload Day")
    ax.set_xlabel("Upload Day")
    ax.set_ylabel("Video Count")

    for idx, row in day_counts.iterrows():
        ax.text(idx, row['video_count'], str(int(row['video_count'])), ha='center', va='bottom', fontsize=9)

    out_path = os.path.join(charts_dir, "video_count_by_upload_day.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)


Saved: eda\charts\upload_hour_vs_day_heatmap.png
Saved: eda\charts\avg_views_by_upload_hour.png
Saved: eda\charts\video_count_by_upload_day.png


## Section 5 — Category Deep Dive

In [5]:
if not df.empty:
    # Grouped bar chart: avg engagement by category
    fig, ax = plt.subplots(figsize=(12, 6))
    cat_eng = df.groupby('category', as_index=False).agg(avg_engagement=('engagement_rate', 'mean'))
    cat_eng = cat_eng.sort_values('avg_engagement', ascending=False)
    sns.barplot(data=cat_eng, x='category', y='avg_engagement', ax=ax, color="#4E79A7")
    ax.set_title("Average Engagement Rate by Category")
    ax.set_xlabel("Category")
    ax.set_ylabel("Avg Engagement Rate (%)")

    for idx, row in cat_eng.iterrows():
        ax.text(idx, row['avg_engagement'], f"{row['avg_engagement']:.3f}", ha='center', va='bottom', fontsize=9)
    out_path = os.path.join(charts_dir, "avg_engagement_by_category.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)

    # Pie chart: share of viral videos per category
    viral_only = df[df['performance_tier'] == 'Viral']
    viral_counts = viral_only['category'].value_counts()
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.pie(viral_counts.values, labels=viral_counts.index, autopct='%1.1f%%', startangle=90)
    ax.set_title("Share of Viral Videos per Category")
    out_path = os.path.join(charts_dir, "viral_share_pie_by_category.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)

    # Scatter plot: view_count vs like_count colored by category
    fig, ax = plt.subplots(figsize=(12, 6))
    df['like_count'] = pd.to_numeric(df['like_count'], errors='coerce')
    sns.scatterplot(data=df, x='view_count', y='like_count', hue='category', ax=ax, alpha=0.7)
    ax.set_title("Views vs Likes by Category")
    ax.set_xlabel("View Count")
    ax.set_ylabel("Like Count")
    corr_vl = float(df['view_count'].corr(df['like_count'])) if df['like_count'].notna().any() else float('nan')
    ax.annotate(f"Corr(view, like)={corr_vl:.3f}", xy=(0.01, 0.98), xycoords='axes fraction',
                ha='left', va='top', fontsize=10, color='black')
    out_path = os.path.join(charts_dir, "view_count_vs_like_count_scatter_by_category.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)


Saved: eda\charts\avg_engagement_by_category.png
Saved: eda\charts\viral_share_pie_by_category.png
Saved: eda\charts\view_count_vs_like_count_scatter_by_category.png


## Section 6 — Channel Analysis

In [6]:
if not df.empty:
    df['subscriber_count'] = pd.to_numeric(df['subscriber_count'], errors='coerce')
    df['view_count'] = pd.to_numeric(df['view_count'], errors='coerce')
    df['engagement_rate'] = pd.to_numeric(df['engagement_rate'], errors='coerce')

    # Top 15 channels by avg engagement (horizontal bar)
    ch_eng = (
        df.groupby(['channel_id', 'channel_name'], as_index=False)
        .agg(avg_engagement_rate=('engagement_rate', 'mean'))
        .sort_values('avg_engagement_rate', ascending=False)
        .head(15)
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(ch_eng['channel_name'].iloc[::-1], ch_eng['avg_engagement_rate'].iloc[::-1], color="#E15759")
    ax.set_title("Top 15 Channels by Avg Engagement Rate")
    ax.set_xlabel("Avg Engagement Rate (%)")
    ax.set_ylabel("Channel")

    for y_idx, val in enumerate(ch_eng['avg_engagement_rate'].iloc[::-1].values):
        ax.text(val, y_idx, f"{val:.3f}", va='center', fontsize=9)

    out_path = os.path.join(charts_dir, "top_15_channels_avg_engagement.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)

    # Scatter: subscriber_count vs avg_video_views
    ch_views = (
        df.groupby(['channel_id', 'channel_name', 'subscriber_count'], as_index=False)
        .agg(avg_video_views=('view_count', 'mean'))
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.scatterplot(data=ch_views, x='subscriber_count', y='avg_video_views', ax=ax, alpha=0.7)
    ax.set_title("Subscriber Count vs Avg Video Views")
    ax.set_xlabel("Subscriber Count")
    ax.set_ylabel("Avg Video Views")

    corr_sv = float(ch_views['subscriber_count'].corr(ch_views['avg_video_views'])) if ch_views['subscriber_count'].notna().any() else float('nan')
    ax.annotate(f"Corr(subscribers, avg views)={corr_sv:.3f}", xy=(0.01, 0.98), xycoords='axes fraction',
                ha='left', va='top', fontsize=10, color='black')
    out_path = os.path.join(charts_dir, "subscriber_count_vs_avg_video_views_scatter.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)

    # Bar: channels with most consistent performance (low stddev)
    ch_consistent = (
        df.groupby(['channel_id', 'channel_name'], as_index=False)
        .agg(stddev_view_count=('view_count', 'std'), video_count=('video_id', 'count') if 'video_id' in df.columns else ('view_count', 'size'))
        .sort_values('stddev_view_count', ascending=True)
        .head(10)
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(ch_consistent['channel_name'].iloc[::-1], ch_consistent['stddev_view_count'].iloc[::-1], color="#59A14F")
    ax.set_title("Most Consistent Channels (Lowest StdDev of Views)")
    ax.set_xlabel("StdDev of View Count")
    ax.set_ylabel("Channel")

    for y_idx, val in enumerate(ch_consistent['stddev_view_count'].iloc[::-1].values):
        ax.text(val, y_idx, f"{val:,.0f}", va='center', fontsize=9)
    out_path = os.path.join(charts_dir, "channels_most_consistent_performance_bar.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)


Saved: eda\charts\top_15_channels_avg_engagement.png
Saved: eda\charts\subscriber_count_vs_avg_video_views_scatter.png
Saved: eda\charts\channels_most_consistent_performance_bar.png


## Section 7 — Correlation Analysis

In [7]:
if not df.empty:
    # Correlation heatmap
    cols = [
        'view_count',
        'like_count',
        'comment_count',
        'duration_seconds',
        'title_length',
        'tag_count',
        'engagement_rate',
        'subscriber_count'
    ]
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    corr = df[cols].corr(numeric_only=True)

    fig, ax = plt.subplots(figsize=(12, 6))

    # Annotate strong correlations only
    annot = corr.round(2).astype(str)
    strong_mask = corr.abs() > 0.5
    annot = np.where(strong_mask, annot, "")

    sns.heatmap(corr, annot=annot, fmt="", cmap="coolwarm", center=0, ax=ax, cbar_kws={'label': 'Correlation'})
    ax.set_title("Correlation Heatmap (Strong Correlations Annotated)")
    out_path = os.path.join(charts_dir, "correlation_heatmap_strong_annotations.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print("Saved:", out_path)


Saved: eda\charts\correlation_heatmap_strong_annotations.png


## Section 8 — Key Insights Summary

In [8]:
if not df.empty:
    insights = []

    # 1) Overall top category by avg engagement
    cat_eng = df.groupby('category', as_index=False).agg(avg_eng=('engagement_rate', 'mean')).sort_values('avg_eng', ascending=False).head(1)
    insights.append(f"Top category by avg engagement rate is {cat_eng['category'].iloc[0]} at {float(cat_eng['avg_eng'].iloc[0]):.4f}%.")

    # 2) Overall top category by avg views
    cat_views = df.groupby('category', as_index=False).agg(avg_views=('view_count', 'mean')).sort_values('avg_views', ascending=False).head(1)
    insights.append(f"Top category by avg views is {cat_views['category'].iloc[0]} at {float(cat_views['avg_views'].iloc[0]):,.0f} views.")

    # 3) Viral share across all videos
    viral_pct = (df['performance_tier'] == 'Viral').mean() * 100.0
    insights.append(f"Viral videos represent {viral_pct:.2f}% of the dataset.")

    # 4) Best day+hour combo overall
    best_time = df.groupby(['upload_day','upload_hour'], as_index=False).agg(avg_eng=('engagement_rate','mean')).sort_values('avg_eng', ascending=False).head(1)
    insights.append(f"Best posting time overall is {best_time['upload_day'].iloc[0]} at {int(best_time['upload_hour'].iloc[0])}:00 with avg engagement {float(best_time['avg_eng'].iloc[0]):.4f}%.")

    # 5) Median engagement rate
    med_eng = float(df['engagement_rate'].median())
    insights.append(f"Median engagement rate is {med_eng:.4f}%.")

    # 6) Engagement efficiency comparison: median likes-to-views ratio
    ratio_like = float(df['like_to_view_ratio'].median()) if 'like_to_view_ratio' in df.columns else np.nan
    insights.append(f"Median like-to-view ratio is {ratio_like:.4f}%.")

    # 7) Long vs Short duration engagement comparison
    dur = df.groupby('duration_category', as_index=False).agg(avg_eng=('engagement_rate','mean')).sort_values('avg_eng', ascending=False)
    if not dur.empty:
        top_dur = dur.iloc[0]
        insights.append(f"Highest avg engagement by duration category is {top_dur['duration_category']} at {float(top_dur['avg_eng']):.4f}%.")

    # 8) Most consistent channels (lowest stddev)
    ch_cons = df.groupby(['channel_id','channel_name'], as_index=False).agg(stddev_views=('view_count','std')).sort_values('stddev_views', ascending=True).head(1)
    insights.append(f"Most consistent channel (lowest view stddev) is {ch_cons['channel_name'].iloc[0]} with stddev {float(ch_cons['stddev_views'].iloc[0]):,.0f}.")

    # 9) Top 10 most viewed cumulative share
    top10_views = df.sort_values('view_count', ascending=False).head(10)['view_count'].sum()
    total_views = df['view_count'].sum()
    share_top10 = (top10_views / total_views) * 100.0 if total_views > 0 else 0.0
    insights.append(f"Top 10 videos contribute {share_top10:.2f}% of total views.")

    # 10) Subscriber vs avg views correlation proxy
    corr = df['subscriber_count'].corr(df['view_count']) if 'subscriber_count' in df.columns else np.nan
    insights.append(f"Correlation proxy between subscriber_count and view_count is {corr:.3f}.")

    print("\n10 Data-backed Insights:")
    for s in insights[:10]:
        print(f"- {s}")
else:
    print("No data available to compute insights.")



10 Data-backed Insights:
- Top category by avg engagement rate is Fitness at 3.7167%.
- Top category by avg views is Fitness at 9,941,981 views.
- Viral videos represent 38.63% of the dataset.
- Best posting time overall is Friday at 4:00 with avg engagement 9.4060%.
- Median engagement rate is 2.7265%.
- Median like-to-view ratio is 2.6282%.
- Highest avg engagement by duration category is Short at 3.3892%.
- Most consistent channel (lowest view stddev) is Pomodo ID with stddev 79.
- Top 10 videos contribute 41.20% of total views.
- Correlation proxy between subscriber_count and view_count is 0.108.
